# Brand-Only Training Pipeline
This notebook extracts datasets from Google Drive, collapses model names into brand names, and trains a fresh PyTorch model from scratch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install split-folders timm

In [ ]:
import os
import zipfile
import shutil
import random
import string

# 1. Define paths (Adjust these if your files are in a specific Drive folder)
vmmrdb_zip = '/content/drive/MyDrive/CoreVision/data/VMMRdb.zip'
car1000_zip = '/content/drive/MyDrive/CoreVision/data/Car-1000-dataset.zip'
manual_brands_zip = '/content/drive/MyDrive/CoreVision/data/manual_brands.zip'

work_dir = '/content/dataset'
master_dir = '/content/dataset/master_brands'
os.makedirs(work_dir, exist_ok=True)
os.makedirs(master_dir, exist_ok=True)

def extract_zip(zip_path, extract_to):
    if os.path.exists(zip_path):
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
    else:
        print(f"Warning: {zip_path} not found.")

extract_zip(vmmrdb_zip, f"{work_dir}/vmmrdb_raw")
extract_zip(car1000_zip, f"{work_dir}/car1000_raw")
extract_zip(manual_brands_zip, f"{work_dir}/manual_brands_raw")

In [ ]:
def random_string(length=6):
    import random
    import string
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

import os
import json
import shutil

# 1. Define strictly allowed brands (normalized to ASCII lowercase)
ALLOWED_BRANDS = [
    'alfaromeo', 'astonmartin', 'audi', 'bentley', 'bmw', 'buick', 'byd',
    'cadillac', 'chery', 'chevrolet', 'chrysler', 'citroen', 'dodge', 'ds',
    'ferrari', 'fiat', 'ford', 'gmc', 'honda', 'hummer', 'hyundai', 'infiniti',
    'isuzu', 'jaguar', 'jeep', 'kia', 'lamborghini', 'landrover', 'lexus',
    'lincoln', 'lotus', 'maserati', 'mazda', 'mclaren', 'mercedesbenz',
    'mg', 'mini', 'mitsubishi', 'nissan', 'peugeot', 'porsche', 'ram',
    'renault', 'rollsroyce', 'saab', 'seat', 'skoda', 'smart', 'subaru',
    'suzuki', 'tesla', 'toyota', 'volkswagen', 'volvo', 'xiaomi', 'opel','cupra'
]

# 2. Chinese mapping so we don't lose the Car-1000 brands
CHINESE_MAPPING = {
    '宝马': 'bmw', '奔驰': 'mercedesbenz', '奥迪': 'audi', '大众': 'volkswagen',
    '丰田': 'toyota', '本田': 'honda', '现代': 'hyundai', '日产': 'nissan',
    '起亚': 'kia', '福特': 'ford', '雪佛兰': 'chevrolet', '标致': 'peugeot',
    '雪铁龙': 'citroen', '雷诺': 'renault', '菲亚特': 'fiat', '斯柯达': 'skoda',
    '沃尔沃': 'volvo', '马自达': 'mazda', '斯巴鲁': 'subaru', '铃木': 'suzuki',
    '三菱': 'mitsubishi', '保时捷': 'porsche', '路虎': 'landrover', '捷豹': 'jaguar',
    '雷克萨斯': 'lexus', '凯迪拉克': 'cadillac', '林肯': 'lincoln', '玛莎拉蒂': 'maserati',
    '宾利': 'bentley', '法拉利': 'ferrari', '兰博基尼': 'lamborghini', '劳斯莱斯': 'rollsroyce',
    '阿斯顿·马丁': 'astonmartin', '阿尔法·罗密欧': 'alfaromeo', '特斯拉': 'tesla',
    '比亚迪': 'byd', '奇瑞': 'chery', '小米汽车': 'xiaomi', '名爵': 'mg',
    'smart': 'smart', 'MINI': 'mini', 'Jeep': 'jeep', '道奇': 'dodge', '克莱斯勒': 'chrysler'
}

def normalize_brand_name(value):
    cleaned = value.lower().strip().replace(' ', '').replace('_', '').replace('-', '')
    cleaned = cleaned.replace('ë', 'e').replace('š', 's')
    return cleaned

def load_car1000_category_brand_map(car1000_root):
    class_info_path = None
    direct_candidate = os.path.join(car1000_root, 'cls_info', 'class_info.json')
    nested_candidate = os.path.join(car1000_root, 'Car-1000-dataset', 'cls_info', 'class_info.json')

    if os.path.exists(direct_candidate):
        class_info_path = direct_candidate
    elif os.path.exists(nested_candidate):
        class_info_path = nested_candidate
    else:
        for root, _, files in os.walk(car1000_root):
            if 'class_info.json' in files:
                class_info_path = os.path.join(root, 'class_info.json')
                break

    if not class_info_path:
        print('  [WARN] Car-1000 class_info.json not found; path matching fallback will be used.')
        return {}

    with open(class_info_path, 'r', encoding='utf-8') as f:
        class_info = json.load(f)

    cat_to_brand = {}

    def assign(cat_id, name_old):
        if not cat_id or not name_old:
            return
        english_part = name_old.split('==', 1)[1] if '==' in name_old else name_old
        english_brand = english_part.split('_', 1)[0]
        brand = normalize_brand_name(english_brand)
        if brand in ALLOWED_BRANDS:
            cat_to_brand[str(cat_id)] = brand
            cat_to_brand[str(cat_id).zfill(4)] = brand

    if isinstance(class_info, list):
        for entry in class_info:
            if isinstance(entry, dict):
                assign(entry.get('id', ''), entry.get('name_from_old', ''))
    elif isinstance(class_info, dict):
        for cat_id, entry in class_info.items():
            if isinstance(entry, dict):
                assign(cat_id, entry.get('name_from_old', ''))
            else:
                assign(cat_id, str(entry))

    return cat_to_brand

def process_brand_model_dir(src_dir, dataset_name, car1000_cat_to_brand=None):
    if not os.path.exists(src_dir):
        print(f"Skipping {src_dir} (not found)")
        return

    print(f"Processing {dataset_name}...")
    count = 0
    failed = 0

    for root, dirs, files in os.walk(src_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                # 3. Clean up the relative path
                rel_path = os.path.relpath(root, src_dir)
                clean_path = normalize_brand_name(rel_path)

                matched_brand = None

                # 4. Car-1000 uses category IDs in paths, so map IDs via class_info.json
                if dataset_name == 'car1000' and car1000_cat_to_brand:
                    path_parts = [p for p in rel_path.split(os.sep) if p and p != '.']
                    cat_candidates = []
                    for i, part in enumerate(path_parts):
                        if part.lower() == 'all_images' and i + 1 < len(path_parts):
                            cat_candidates.append(path_parts[i + 1])
                    if path_parts:
                        cat_candidates.append(path_parts[0])
                    if len(path_parts) > 1:
                        cat_candidates.append(path_parts[1])

                    for cat_id in cat_candidates:
                        matched_brand = car1000_cat_to_brand.get(str(cat_id)) or car1000_cat_to_brand.get(str(cat_id).zfill(4))
                        if matched_brand:
                            break

                # 5. Check if a Chinese brand name is anywhere in the original path
                if not matched_brand:
                    for zh_brand, en_brand in CHINESE_MAPPING.items():
                        if zh_brand in rel_path:
                            matched_brand = en_brand
                            break

                # 6. Check if an English brand name is in the cleaned path
                if not matched_brand:
                    for brand in sorted(ALLOWED_BRANDS, key=len, reverse=True):
                        if brand in clean_path:
                            matched_brand = brand
                            break

                # 7. Fallback check for manual datasets if it's <brand>/<model>
                if not matched_brand and dataset_name == 'manual':
                    parts = rel_path.split(os.sep)
                    if len(parts) >= 1:
                        possible = normalize_brand_name(parts[0])
                        if possible in ALLOWED_BRANDS:
                            matched_brand = possible

                # 8. ONLY keep the image if it matches one of our ALLOWED_BRANDS
                if matched_brand and matched_brand in ALLOWED_BRANDS:
                    dest_brand_dir = os.path.join(master_dir, matched_brand)
                    os.makedirs(dest_brand_dir, exist_ok=True)

                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(dest_brand_dir, f"{dataset_name}_{matched_brand}_{random_string()}_{file}")

                    try:
                        shutil.copy2(src_file, dest_file)
                        count += 1
                    except Exception as e:
                        # FIX: Log copy failures instead of silently swallowing them
                        print(f"  [WARN] Failed to copy {src_file}: {e}")
                        failed += 1

    print(f"Copied {count} images for {dataset_name} ({failed} failed)")

# Process all three sources
car1000_cat_to_brand = load_car1000_category_brand_map(f"{work_dir}/car1000_raw")
if car1000_cat_to_brand:
    print(f"Car-1000 category mappings loaded: {len(car1000_cat_to_brand)}")
process_brand_model_dir(f"{work_dir}/vmmrdb_raw", 'vmmrdb')
process_brand_model_dir(f"{work_dir}/car1000_raw", 'car1000', car1000_cat_to_brand)
process_brand_model_dir(f"{work_dir}/manual_brands_raw", 'manual')

brands = os.listdir(master_dir)
print(f"\nTotal unique allowed brands found: {len(brands)}")
print("Sample brands:", brands[:20])

# FIX: Print per-class image counts so you can spot imbalance early
print("\nPer-brand image counts:")
brand_counts = {}
for brand in sorted(brands):
    brand_path = os.path.join(master_dir, brand)
    n = len([f for f in os.listdir(brand_path) if os.path.isfile(os.path.join(brand_path, f))])
    brand_counts[brand] = n
    print(f"  {brand:20s}: {n:6d} images")
print(f"\nMin: {min(brand_counts.values())}  Max: {max(brand_counts.values())}  Ratio: {max(brand_counts.values())/max(min(brand_counts.values()),1):.1f}x")
print(brands)

In [ ]:
import splitfolders

print("Splitting dataset into train and validation sets...")
split_dir = '/content/dataset/split'
splitfolders.ratio(master_dir, output=split_dir, seed=42, ratio=(.8, .2), group_prefix=None)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import WeightedRandomSampler
import timm
import numpy as np

# Training configurations
# FIX: Increased batch size from 64 -> 128 to better utilise L4's 24GB VRAM
BATCH_SIZE = 128
EPOCHS = 15
LEARNING_RATE = 0.001

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    # FIX: Added Resize(256) + CenterCrop(224) to match EfficientNet pretraining convention
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {
    x: datasets.ImageFolder(os.path.join(split_dir, x), data_transforms[x])
    for x in ['train', 'val']
}

# FIX: WeightedRandomSampler to handle class imbalance
# Rare brands (lotus, mclaren, astonmartin) will be sampled more frequently
train_targets = image_datasets['train'].targets
class_counts = np.bincount(train_targets)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_targets]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

dataloaders = {
    # FIX: shuffle=False when using sampler (sampler handles randomisation)
    # FIX: num_workers raised from 2 -> 6 to prevent GPU starvation on L4
    # FIX: pin_memory=True for faster CPU->GPU transfers
    'train': torch.utils.data.DataLoader(
        image_datasets['train'], batch_size=BATCH_SIZE,
        sampler=sampler, num_workers=6, pin_memory=True
    ),
    'val': torch.utils.data.DataLoader(
        image_datasets['val'], batch_size=BATCH_SIZE,
        shuffle=False, num_workers=6, pin_memory=True
    ),
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
num_classes = len(class_names)

print(f"Classes: {num_classes}")
print(f"Train images: {dataset_sizes['train']}, Val images: {dataset_sizes['val']}")

In [ ]:
# Initialize EfficientNetV2-S via timm
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
# FIX: Replaced StepLR (harsh single drop) with CosineAnnealingLR (smooth decay over all epochs)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# FIX: GradScaler for AMP mixed-precision training
scaler = torch.cuda.amp.GradScaler()

In [ ]:
import copy
import time

def train_model(model, criterion, optimizer, scheduler, scaler, num_epochs=EPOCHS):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad()

                # FIX: AMP autocast for faster fp16 forward pass on L4
                with torch.set_grad_enabled(phase == 'train'):
                    with torch.cuda.amp.autocast(enabled=(phase == 'train')):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                    if phase == 'train':
                        # FIX: Scaled backward pass for AMP stability
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

model = train_model(model, criterion, optimizer, scheduler, scaler, num_epochs=EPOCHS)

In [ ]:
# Save the trained model to Google Drive
save_path = '/content/drive/MyDrive/brand_classifier_best.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

# Also save the class names mapping
import json
with open('/content/drive/MyDrive/brand_classes.json', 'w') as f:
    json.dump(class_names, f)
print("Class names saved to Drive as well.")